Importantions of libraries

In [1]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf
from typing import Optional
import requests
from datetime import datetime, timedelta
from abc import ABC, abstractmethod
import logging
from typing import List, Dict, Optional, Union
from dataclasses import dataclass, field


In [ ]:
(datetime.now() + timedelta(days=7)).strftime('%Y-%m-%d')

'2026-04-08'

Class Stocks

In [ ]:
@dataclass
class Asset(ABC):
    symbol: str
    _prices: Optional[float] = None

    @property
    @abstractmethod
    def prices(self) -> float:
        pass

@dataclass
class Crypto(Asset):
    period: str = "1d"
    interval: str = "1m"
    provider: Provider = None

    def __post_init__(self):
        if self.provider is None:
            self.provider = YahooProvider()

    def _prices(self) -> float:
        if self._prices is None:
            self._prices = self.provider.get_price(self.symbol)
        return self._prices

    def get_close_series(self) -> pd.Series:
        return self._prices[self.symbol]['Close']

    def get_current_price(self) -> float:
        return self._prices[self.symbol]['Close'][-1]

@dataclass
class Indicator(ABC):
    @abstractmethod
    def calculate(self, asset: Asset):
        pass

@dataclass
class SimpleMovingAverage(Indicator):
    period: int = 20

    def calculate(self, asset: Asset):
        close = asset.get_close_series()
        return close.rolling(self.period).mean()

@dataclass
class ExponentialMovingAverage(Indicator):
    period: int = 20

    def calculate(self, asset: Asset):
        close = asset.get_close_series()
        return close.ewm(span=self.period).mean()

@dataclass
class Rsi(Indicator):
    period: int = 14

    def calculate(self, asset: Asset):
        close = asset.get_close_series()
        delta = close.diff()
        gain = (delta.where(delta > 0, 0)).rolling(self.period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(self.period).mean()
        loss = loss.replace(0, np.nan)
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

@dataclass
class StochRsi(Indicator):
    period: int = 14
    smooth: int = 3
    rsi: Rsi = field(default_factory=Rsi)

    def calculate(self, asset: Asset):
        rsi_series = self.rsi.calculate(asset)
        rsi_low = rsi_series.rolling(self.period).min()
        rsi_high = rsi_series.rolling(self.period).max()
        denom = (rsi_high - rsi_low).replace(0, np.nan)
        k_raw = (rsi_series - rsi_low) / denom * 100
        k = k_raw.rolling(window=self.smooth).mean() if self.smooth and self.smooth > 1 else k_raw
        d = k.rolling(window=self.smooth).mean() if self.smooth and self.smooth > 1 else k
        return pd.DataFrame({
            "k": k,
            "d": d
            }).dropna()

@dataclass
class BollingerBands(Indicator):
    window=20
    std_dev=2

    def calculate(self, asset: Asset):
        price = asset.get_close_series()
        middle = price.rolling(self.window).mean()
        std = price.rolling(self.window).std()
        upper = middle + self.std_dev * std
        lower = middle - self.std_dev * std
        bb_percent_b = (price - lower) / (upper - lower)
        return bb_percent_b
@dataclass

class Macd(Indicator):
    fast: int = 12
    slow: int = 26
    signal: int = 9

    def calculate(self, asset: Asset):
        close = asset.get_close_series()
        exp1 = close.ewm(span=self.fast).mean()
        exp2 = close.ewm(span=self.slow).mean()
        macd = exp1 - exp2
        signal = macd.ewm(span=self.signal).mean()
        histogram = macd - signal
        return pd.DataFrame({
        "macd": macd,
        "signal": signal,
        "histogram": histogram
        })

@dataclass
class Provider(ABC):
    @abstractmethod
    def get_price(self, symbol: str) -> float:
        pass

@dataclass
class YahooProvider(Provider):

    def get_price(self, symbol: str) -> float:
        return yf.download(symbol, period=self.period, interval=self.interval)

class Calculations:

    @staticmethod
    def log_return(prices):
        return pd.Series([np.log(prices[i]/prices[i-1]) for i in range(1, len(prices))], index=prices.index[1:])

    @staticmethod
    def annualized_sharpe(returns, period=252, rf=0.01, window=252):
        rolling_mean = returns.rolling(window=window).mean() * period
        rolling_std = returns.rolling(window=window).std() * np.sqrt(period)
        sharpe_ratio = (rolling_mean - rf) / rolling_std
        return sharpe_ratio

    @staticmethod
    def annualized_sortino(returns, period=252, rf=0.01, window=252):
        mar = rf / period
        excess = returns - mar
        downside = np.minimum(0, excess)
        downside_deviation = downside.rolling(window=window).std() * np.sqrt(period)
        rolling_mean = excess.rolling(window=window).mean() * period
        sortino_ratio = rolling_mean / downside_deviation
        sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
        return sortino_ratio

@dataclass
class SharpeRatio(Indicator):
    period: int = 252
    window: int = 252
    rf: float = 0.01

    def calculate(self, asset: Asset):
        close_prices = asset.get_close_series()
        returns = Calculations.log_return(close_prices)
        sharpe_ratio = Calculations.annualized_sharpe(returns, self.period, self.window, self.rf)
        return sharpe_ratio

@dataclass
class SortinoRatio(Indicator):
    period: int = 252
    window: int = 252
    rf: float = 0.01

    def calculate(self, asset: Asset):
        close_prices = asset.get_close_series()
        returns = close_prices.pct_change().dropna()
        sortino_ratio = Calculations.annualized_sortino(returns, self.period, self.window, self.rf)
        return sortino_ratio


@dataclass
class Btc(Crypto):
    pass

In [ ]:
class Stock:

    def __init__(self, symbol: str, date_init: str, date_end: str = (datetime.now() + timedelta(days=7)).strftime('%Y-%m-%d'), interval: str = "1d", thresholds=None):
        self.symbol = symbol
        self.date_init = date_init
        self.date_end = date_end or datetime.now() + timedelta(days=7).strftime('%Y-%m-%d')
        self.interval = interval
        self._df: Optional[pd.DataFrame] = None
        self.thresholds = thresholds or {
            "BTC-USD": {
                "sharpe_top": 2.05,
                "sortino_top": 5.65,
                "sharpe_bottom": -1.1,
                "sortino_bottom": -1.7,
            },
            "SPY": {
                "sharpe_top": 2.2,
                "sortino_top": 5,
                "sharpe_bottom": -0.5,
                "sortino_bottom": -1.1,
            }
        }
    @property
    def df(self) -> pd.DataFrame:
        if self._df is None:  # Só baixa se não tiver dados
            self._df = self.create_dataframe()
        return self._df

    def __str__(self):
        return f"Stock(symbol='{self.symbol}', date_init='{self.date_init}', date_end='{self.date_end}', interval='{self.interval}')"

    def create_dataframe(self):
        return yf.download(self.symbol, start=self.date_init, end=self.date_end, interval=self.interval).dropna().copy()

    def create_dataframe_close(self):
        close = self.df["Close"]
        if isinstance(close, pd.DataFrame):
            if self.symbol in close.columns:
                close = close[self.symbol]
            else:
                close = close.iloc[:, 0]
        return close

    def create_dataframe_low(self):
        low = self.df["Low"]
        if isinstance(low, pd.DataFrame):
            if self.symbol in low.columns:
                low = low[self.symbol]
            else:
                low = low.iloc[:, 0]
        return low

    def create_dataframe_high(self):
        high = self.df["High"]
        if isinstance(high, pd.DataFrame):
            if self.symbol in high.columns:
                high = high[self.symbol]
            else:
                high = high.iloc[:, 0]
        return high

    def normalize(self):
        close = self.create_dataframe_close()
        return close / close.iloc[0]

    def log_return(self):
        close = self.create_dataframe_close()
        return np.log(close / close.shift(1)).dropna()

    def return_investment(self):
        acumulated_returns = self.create_dataframe_close().pct_change().dropna()
        return (1 + acumulated_returns).cumprod()

    def sharpe_ratio(self):
        rf = 0.01
        if self.interval == "1mo":
            rolling_mean = self.log_return().rolling(window=6).mean() * 12
            rolling_std = self.log_return().rolling(window=6).std() * np.sqrt(12)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio.reindex(self.df.index)
        elif self.interval == "1wk":
            rolling_mean = self.log_return().rolling(window=60).mean() * 52
            rolling_std = self.log_return().rolling(window=60).std() * np.sqrt(52)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio.reindex(self.df.index)

        elif self.interval == "1d":
            rolling_mean = self.log_return().rolling(window=252).mean() * 252
            rolling_std = self.log_return().rolling(window=252).std() * np.sqrt(252)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio.reindex(self.df.index)
        else:
            return 0

    def sortino_ratio(self):
        returns = self.create_dataframe_close().pct_change().dropna()
        rf = 0.01
        if self.interval == "1mo":
            mar = rf / 12
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=14).std() * np.sqrt(12)
            rolling_mean = excess.rolling(window=14).mean() * 12
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio.reindex(self.df.index)
        elif self.interval == "1wk":
            mar = rf / 52
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=60).std() * np.sqrt(52)
            rolling_mean = excess.rolling(window=60).mean() * 52
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio.reindex(self.df.index)
        if self.interval == "1d":
            mar = rf / 252
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=252).std() * np.sqrt(252)
            rolling_mean = excess.rolling(window=252).mean() * 252
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio.reindex(self.df.index)
        else:
            return 0

    def compute_masks(self):
        shp = self.sharpe_ratio()
        srt = self.sortino_ratio()

        th = self.thresholds[self.symbol]

        masks = {
            "sortino_top": srt > th["sortino_top"],
            "sharpe_top": shp > th["sharpe_top"],
            "sharpe_sortino_top": (srt >= th["sortino_top"]) & (shp > th["sharpe_top"]),

            "sortino_reversion_down": (shp < 1.2) & (srt >= th["sortino_top"]),

            "sharpe_bottom": shp < th["sharpe_bottom"],
            "sortino_bottom": srt < th["sortino_bottom"],
            "sharpe_sortino_bottom": (srt < th["sortino_bottom"]) & (shp < th["sharpe_bottom"]),

            "sortino_reversion_up": (shp > 0) & (srt <= th["sortino_bottom"]),
        }
        return masks

    def ema(self, period: int):
        close = self.create_dataframe_close()
        return close.ewm(span=period, adjust=False).mean()

    def macd(self, fast=12, slow=26, signal=9):
        macd_line = self.ema(fast) - self.ema(slow)
        signal_line = macd_line.ewm(span=signal, adjust=False).mean()
        histogram = macd_line - signal_line
        return macd_line, signal_line, histogram

    def bb_percent(self, window=20, std=2):
        close = self.create_dataframe_close()
        midle = close.rolling(window).mean()
        upper = midle + (close.rolling(window).std() * std)
        lower = midle - (close.rolling(window).std() * std)
        return (close - lower) / (upper - lower)

    def rsi(self, period: int = 14):
        close = self.create_dataframe_close()
        delta = close.diff()
        gain = (delta.where(delta > 0, 0)).rolling(period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi


    def stoch_rsi(self, rsi_period: int = 14, stoch_period: int = 14, smooth_k: int = 3, smooth_d: int = 3):
        rsi = self.rsi(rsi_period)

        rsi_low = rsi.rolling(stoch_period).min()
        rsi_high = rsi.rolling(stoch_period).max()

        denom = (rsi_high - rsi_low).replace(0, np.nan)
        k_raw = (rsi - rsi_low) / denom * 100

        k = k_raw.rolling(window=smooth_k).mean() if smooth_k and smooth_k > 1 else k_raw
        d = k.rolling(window=smooth_d).mean() if smooth_d and smooth_d > 1 else k

        return k.dropna(), d.dropna()

@dataclass
class Signal(ABC):

    @abstractmethod
    def get_sinal_top(self) -> bool:
        pass

    @abstractmethod
    def get_signal_bottom(self) -> bool:
        pass


In [38]:
# Exemplo de uso
btc_w = Stock("BTC-USD", "2018-01-01", interval="1wk")
vix_w = Stock("^VIX", "2018-01-01", interval="1wk")
spy_w = Stock("SPY", "2018-01-01", interval="1wk")
eth_w = Stock("ETH-USD", "2018-01-01", interval="1wk")

In [39]:
eth_w.thresholds['ETH-USD'] = {'sharpe_top': 1.7, 'sortino_top': 5.65, 'sharpe_bottom': -1.1, 'sortino_bottom': -1.7}
vix_w.thresholds['^VIX'] = {'sharpe_top': 0.5, 'sortino_top': 2, 'sharpe_bottom': -0.5, 'sortino_bottom': -1}

In [40]:
btc_w.df

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2018-01-01,16477.599609,17712.400391,13154.700195,14112.200195,123814400000
2018-01-08,13772.000000,16537.900391,13105.900391,16476.199219,106022199296
2018-01-15,11600.099609,14445.500000,9402.290039,13767.299805,97932879872
2018-01-22,11786.299805,12040.299805,10129.700195,11633.099609,64691999232
2018-01-29,8277.009766,11875.599609,7796.490234,11755.500000,60810019840
...,...,...,...,...,...
2026-03-09,72789.914062,73927.328125,65858.007812,65969.585938,301054621121
2026-03-16,67845.210938,75988.398438,67372.875000,72798.171875,285450089323


In [41]:
btc_mask = btc_w.compute_masks()
spy_mask = spy_w.compute_masks()
eth_mask = eth_w.compute_masks()
vix_mask = vix_w.compute_masks()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [42]:
fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Scatter(
    x=btc_w.ema(20).index,
    y=btc_w.ema(20)
))
fig.add_trace(go.Scatter(
    x=btc_w.ema(50).index,
    y=btc_w.ema(50)
))
fig.add_trace(go.Scatter(
    x=btc_w.ema(100).index,
    y=btc_w.ema(100)
))
fig.add_trace(go.Scatter(
    x=btc_w.ema(200).index,
    y=btc_w.ema(200)
))
fig.show()

In [43]:
sharpe_top_dates_spy = spy_w.sharpe_ratio().index[spy_mask["sharpe_top"]]
sharpe_top_values_spy = spy_w.sharpe_ratio()[spy_mask["sharpe_top"]]

fig_spy_ss = make_subplots(rows=2, cols=1)

fig_spy_ss.add_trace(go.Scatter(
    x=spy_w.sharpe_ratio().index,
    y=spy_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_spy_ss.add_trace(go.Bar(
    x=spy_w.sortino_ratio().index,
    y=spy_w.sortino_ratio(),
    marker_color=spy_w.sortino_ratio())
    , row=1, col=1)
fig_spy_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_spy,
    y=sharpe_top_values_spy,
    mode='markers',
    marker=dict(size=sharpe_top_values_spy*5, color='red', symbol='diamond'),
), row=1, col=1)

fig_spy_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_spy_ss

In [44]:
sharpe_top_dates_btc = btc_w.sharpe_ratio().index[btc_mask["sharpe_top"]]
sharpe_top_values_btc = btc_w.sharpe_ratio()[btc_mask["sharpe_top"]]

fig_btc_ss = make_subplots(rows=1, cols=1)

fig_btc_ss.add_trace(go.Scatter(
    x=btc_w.sharpe_ratio().index,
    y=btc_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_btc_ss.add_trace(go.Bar(
    x=btc_w.sortino_ratio().index,
    y=btc_w.sortino_ratio(),
    marker_color=btc_w.sortino_ratio())
    , row=1, col=1)
fig_btc_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_btc,
    y=sharpe_top_values_btc,
    mode='markers',
    marker=dict(size=sharpe_top_values_btc*5, color='red', symbol='diamond'),
), row=1, col=1)

fig_btc_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_btc_ss.show()

In [45]:
sharpe_top_dates_vix = vix_w.sharpe_ratio().index[vix_mask["sharpe_top"]]
sharpe_top_values_vix = vix_w.sharpe_ratio()[vix_mask["sharpe_top"]]

fig_vix_ss = make_subplots(rows=2, cols=1)

fig_vix_ss.add_trace(go.Scatter(
    x=vix_w.sharpe_ratio().index,
    y=vix_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_vix_ss.add_trace(go.Bar(
    x=vix_w.sortino_ratio().index,
    y=vix_w.sortino_ratio(),
    marker_color=vix_w.sortino_ratio())
    , row=1, col=1)
fig_vix_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_vix,
    y=sharpe_top_values_vix,
    mode='markers',
    marker=dict(size=sharpe_top_values_vix*20, color='red', symbol='diamond'),
), row=1, col=1)

fig_vix_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_vix_ss

In [46]:
sharpe_top_dates_eth = eth_w.sharpe_ratio().index[eth_mask["sharpe_top"]]
sharpe_top_values_eth = eth_w.sharpe_ratio()[eth_mask["sharpe_top"]]

fig_eth_ss = make_subplots(rows=2, cols=1)

fig_eth_ss.add_trace(go.Scatter(
    x=eth_w.sharpe_ratio().index,
    y=eth_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_eth_ss.add_trace(go.Bar(
    x=eth_w.sortino_ratio().index,
    y=eth_w.sortino_ratio(),
    marker_color=eth_w.sortino_ratio())
    , row=1, col=1)
fig_eth_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_eth,
    y=sharpe_top_values_eth,
    mode='markers',
    marker=dict(size=sharpe_top_values_eth*4, color='red', symbol='diamond'),
), row=1, col=1)

fig_eth_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_eth_ss

In [47]:
macd_line, signal_line, histogram = btc_w.macd()

fig3 = make_subplots(rows=1, cols=1)

fig3.add_trace(go.Bar(
    x=histogram.index,
    y=histogram,
    name='Histogram',
    marker_color=histogram
))

fig3.add_trace(go.Scatter(
    x=macd_line.index,
    y=macd_line,
    name='MACD Line',
    line=dict(color='#2962FF')
))

fig3.add_trace(go.Scatter(
    x=signal_line.index,
    y=signal_line,
    name='Signal Line',
    line=dict(color='#FF6D00')
))

fig3.update_layout(title='MACD - BTC', xaxis_title='Date', yaxis_title='Value', template=('plotly_dark'))
fig3.show()

In [48]:
fig4 = make_subplots(rows=1, cols=1)

fig4.add_trace(go.Scatter(
    x=btc_w.bb_percent().index,
    y=btc_w.bb_percent()
))
fig4.show()

In [49]:
stoch_k, stoch_d = btc_w.stoch_rsi()
rsi = btc_w.rsi()

fig5 = make_subplots(rows=1, cols=1)

fig5.add_trace(go.Scatter(
    x=stoch_d.index,
    y=stoch_d))

fig5.add_trace(go.Scatter(
    x=rsi.index,
    y=rsi))

fig5.show()

RSI + STOCHASTICO OS DOIS JUNTOS NAS EXTREMIDADES É INVERSAO DE TENDENCIA